### GPU/CUDA acceleration example
The CUDA kernel accelerates simulation of multiple parameter sets (particles) for 1- or 2-timescale Ornstein-Uhlenbeck (OU) processes in parallel. For each particle, it:
- Simulates multiple trials of the OU process.
- Computes the autocorrelation function (ACF) for each trial.
- Accumulates statistics such as trial-wise ACFs, average ACF, and distance to the observed ACF.
Output behavior is controlled by a bitmask `mode`, allowing selective retrieval of desired results (e.g., distances only, full trial data, or ACFs).

#### Imports

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import sys
from scipy.io import loadmat

#### Add local modules to system path for import resolution

In [2]:
sys.path.extend(['.','./abcTau'])

#### Local project imports

In [3]:
from abcTau.gpu.pmc import pmc_abc                    # Population Monte Carlo ABC driver
from abcTau.gpu.model import CUDAModel                # Custom model wrapper using CUDA acceleration
from abcTau.gpu.config import ABCConfig               # Configuration dataclass for all model/ABC settings

#### Load synthetic observed data from a ground-truth 1-OU process
This is the "real" data we will try to match via simulation and ABC inference

In [4]:
real_data = np.load('./example_data/OU_tau20_mean0_var1_rawData.npy')
n_trials, timesteps = real_data.shape  # Get data dimensions (n_trials x timesteps)

#### Define simulation output mode using bitmask flags
Use bitwise OR (`|`) to combine multiple outputs.

Available flags:
- `OUTPUT_DISTANCE`   – compute distance to real ACF
- `OUTPUT_AVG_ACF`    – output trial-averaged ACF
- `OUTPUT_TRIAL_ACF`  – output trial-by-trial ACFs
- `OUTPUT_SIM_DATA`   – output raw simulated data

Example: `mode = OUTPUT_DISTANCE | OUTPUT_AVG_ACF | OUTPUT_TRIAL_ACF`

In [5]:
OUTPUT_DISTANCE  = 1 << 0  # Compute distance between simulated and real ACF
OUTPUT_AVG_ACF   = 1 << 1  # Return trial-averaged ACF per particle
OUTPUT_TRIAL_ACF = 1 << 2  # Return ACFs for each individual trial
OUTPUT_SIM_DATA  = 1 << 3  # Return raw simulated data per trial
mode = OUTPUT_DISTANCE | OUTPUT_AVG_ACF | OUTPUT_TRIAL_ACF | OUTPUT_SIM_DATA # only output distances in this example

#### Define the generative model
We are modeling the observed data using Ornstein-Uhlenbeck (OU) processes. Here we choose how many OU processes to include in the simulation:
- `n_ou = 1` uses a single-timescale OU process (simpler model)
- `n_ou = 2` would model data as a mixture of two OU processes (for multi-timescale dynamics)

In [6]:
n_ou = 1  # number of OU processes in the generative model

#### Prior over model parameters
For the 1-OU model, there is a single time constant (tau). This prior assumes tau ~ Uniform(0, 100).

In [7]:
tau_min = 0.0
tau_max = 100.0
prior = [stats.uniform(loc=tau_min, scale=tau_max-tau_min)]

#### ABC Configuration
This dataclass holds all relevant simulation, prior, and ABC settings

In [8]:
cfg = ABCConfig(
    real_data       = real_data,    # Raw observed data
    n_trials        = n_trials,     # Number of trials to simulate
    timesteps       = timesteps,    # Number of time steps per trial
    deltaT          = 1,            # Time resolution (ms)
    binSize         = 1,            # Binning size for analysis (ms)
    maxTimeLag      = 50,           # Max lag (in time bins) for ACF
    prior           = prior,        # Prior over parameters
    mode            = mode,         # Output modes from the CUDA kernel (bitmask)
    epsilon_0       = 1.0,          # Initial ABC acceptance threshold
    min_samples     = 100,          # Minimum accepted samples per iteration
    steps           = 60,           # Max number of PMC iterations
    n_ou            = n_ou,         # OU model type (1 or 2 processes)
    epsilon_floor   = 0.0001,         # Terminate if epsilon drops below this
    output_path     = '../outputs.mat'
)

#### Model instantiation and CUDA setup
This object wraps the simulation kernel and prepares input/output buffers

In [9]:
model = CUDAModel(cfg)
model.load_cuda()  # Load the shared CUDA library and bind the simulation function

#### Run the PMC-ABC inference loop
This will iteratively refine the posterior using simulation + distance rejection

In [10]:
pmc_abc(model)


📦 PMC Step 1/60
----------------------------------------
  ε (threshold)     : 1.00000
  Accepted samples  : 100 / 100
  Acceptance rate   : 100.00%
  Effective Sample Size (ESS): 100.00
  τ² (covariance scale)      : 1547.824040466645
[PMC] Cumulative elapsed time: 0.72 sec

📦 PMC Step 2/60
----------------------------------------
  ε (threshold)     : 0.11430
  Accepted samples  : 128 / 200
  Acceptance rate   : 64.00%
  Effective Sample Size (ESS): 125.31
  τ² (covariance scale)      : (1, 1)
[PMC] Cumulative elapsed time: 1.92 sec

📦 PMC Step 3/60
----------------------------------------
  ε (threshold)     : 0.08494
  Accepted samples  : 124 / 200
  Acceptance rate   : 62.00%
  Effective Sample Size (ESS): 122.46
  τ² (covariance scale)      : (1, 1)
[PMC] Cumulative elapsed time: 3.13 sec

📦 PMC Step 4/60
----------------------------------------
  ε (threshold)     : 0.05486
  Accepted samples  : 121 / 200
  Acceptance rate   : 60.50%
  Effective Sample Size (ESS): 119.71
  τ² (

#### Load outputs

In [11]:
outputs = loadmat(model.cfg.output_path)

#### Assess goodness of fit via posterior predictive check

This section evaluates the goodness of fit of the inferred model using a posterior predictive check (PPC) based on the maximum a posteriori (MAP) estimate.

Steps:
- We compute the MAP estimate from the weighted ABC posterior using kernel density estimation (KDE).
- We then simulate new synthetic data using the MAP parameters only.
- To assess the quality of the fit, we compare the distribution of the simulated data to that of the observed data.

The comparison is performed directly on the raw time series values, rather than on summary statistics like autocorrelation functions (ACFs), in order to test the model's ability to capture the full distributional structure of the data beyond the features it was trained to match.

To quantify the similarity between the real and synthetic data distributions, we use the Kolmogorov–Smirnov (KS) statistic, which measures the maximum difference between their empirical cumulative distribution functions.
- The KS statistic ranges from 0 to 1, where lower values indicate better agreement between the two datasets.
- We focus on the KS statistic itself to assess fit quality.

While the KS test also yields a p-value, we do not interpret it here. This is because, with large sample sizes such as in this example, even small and practically irrelevant differences between the distributions can produce very low p-values, making them uninformative in this context. Instead, the KS statistic offers a more meaningful and scale-independent summary of distributional fit.

In [12]:
from scipy.stats import gaussian_kde

def MAP_estimate(posterior_thetas, posterior_weights, n_points):
    # Fit weighted KDE to the posterior samples
    kde = gaussian_kde(posterior_thetas, weights=posterior_weights)

    # Generate evaluation points uniformly within parameter bounds
    eval_points = np.vstack([
        np.random.uniform(low=np.min(posterior_thetas[i]),
                            high=np.max(posterior_thetas[i]),
                            size=n_points)
        for i in range(model.cfg.n_params)
    ])

    # Evaluate posterior density at sampled points
    densities = kde(eval_points)

    # Get point with maximum density
    theta_map = eval_points[:, np.argmax(densities)]

    return theta_map

In [13]:
posterior_weights = outputs['weights'][-1] # get weights of final iteration
posterior_weights = posterior_weights[~np.isnan(posterior_weights)] # remove trailing filler NaNs

# same for parameters
posterior_thetas = outputs['accepted_theta'][-1]
posterior_thetas = posterior_thetas[~np.isnan(posterior_thetas)]

n_points = 1_000_000

theta_map = MAP_estimate(posterior_thetas, posterior_weights, n_points)

print(f"MAP tau = {theta_map[0]:.2f} ms (ground truth = 20 ms).")

MAP tau = 19.86 ms (ground truth = 20 ms).


In [14]:
from abcTau.gpu.simulator import run_cuda_simulation

model_ppc = model
model_ppc.cfg.mode = OUTPUT_SIM_DATA
_, _, _, syn_data = run_cuda_simulation(model_ppc, np.atleast_2d(np.atleast_2d(theta_map)))

In [15]:
from scipy.stats import ks_2samp

syn_data_flat = syn_data.flatten()
real_data_flat = real_data.flatten()

ks_statistic, p_value = ks_2samp(real_data_flat, syn_data_flat) # p should be << 1 i.e. generative model is a good fit to the data

print(f"KS statistic = {ks_statistic:.2f} out of 1")

KS statistic = 0.01 out of 1
